# MOZYME en GPU: optimización y dinámica molecular de proteínas

Flujo completo sobre un PDB: preparación (limpieza, hidrógenos, comprobación química), optimización de geometría,
equilibrado NVT y producción NVE, con el SCF de MOZYME en la GPU (binario compilado de `github.com/juvenalyosa/mopac_gpu`).

**Qué hay y qué no**

| | Disponible | Cómo |
|---|---|---|
| NVE (microcanónico) | sí | `DRC TEMPERATURE=T`: velocidades iniciales de Maxwell-Boltzmann a T, luego energía constante |
| NVT (canónico) | sí | `DRC TEMPERATURE=T BUSSI=τ`: termostato de Bussi (reescalado estocástico de velocidades, ensamble canónico exacto), τ en fs |
| NPT | no | MOZYME trata una molécula finita; con solvente implícito no hay caja ni presión |
| Solvente implícito | sí | COSMO: `EPS=78.4` (agua). La celda 9 mide si corre en la GPU o vuelve a la CPU |
| Solvente explícito periódico | no en GPU | MOZYME periódico usa la CPU |

El integrador de `DRC` ajusta el paso de tiempo (del orden de 0.1 fs con hidrógenos); la tabla se imprime cada
`INTERVAL_FS` femtosegundos y la trayectoria completa queda en `nvt.xyz` / `nve.xyz`. La columna ERROR es el error de
integración (la energía intercambiada con el termostato se contabiliza aparte), y debe quedar pequeña frente a la
energía cinética.

Orden: celdas 1 a 3 (GPU, código, compilación, ~10 min), 4 (parámetros), 5 a 10.

## 1. GPU

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2. Código fuente

In [ ]:
from pathlib import Path
import shutil, subprocess

REPO_URL = 'https://github.com/juvenalyosa/mopac_gpu.git'
BRANCH = 'main'
CONTENT = Path('/content')
SRC = CONTENT / 'mopac_src'
BUILD = CONTENT / 'mopac_build'
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(SRC)], check=True)
print(subprocess.run(['git', '-C', str(SRC), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 3. Compilar MOPAC con GPU

In [ ]:
import subprocess, shutil

def run(cmd, **kw):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    return subprocess.run([str(c) for c in cmd], check=True, **kw)

pkgs = ['cmake', 'gfortran', 'ninja-build', 'libblas-dev', 'liblapack-dev']
for attempt in range(2):
    subprocess.run(['apt-get', 'update', '-qq'], check=False)
    r = subprocess.run(['apt-get', 'install', '-y', '-qq', '--no-install-recommends', *pkgs],
                       check=False, capture_output=True, text=True)
    if r.returncode == 0:
        break
    if attempt == 1:
        raise SystemExit('apt-get install failed twice')
if BUILD.exists():
    shutil.rmtree(BUILD)
cmake_cmd = ['cmake', '-S', SRC, '-B', BUILD, '-GNinja', '-DGPU=ON', '-DTESTS=OFF', '-DCMAKE_BUILD_TYPE=RelWithDebInfo']
try:
    run(cmake_cmd + ['-DCUDA_ARCHS=native'])
except subprocess.CalledProcessError:
    shutil.rmtree(BUILD, ignore_errors=True)
    run(cmake_cmd + ['-DCUDA_ARCHS=all'])
run(['cmake', '--build', BUILD, '--target', 'mopac', '--parallel', '2'])
MOPAC = BUILD / 'mopac'
assert MOPAC.exists(), 'mopac executable not built'
print('OK:', MOPAC)

## 4. Parámetros

`PDB_ID` se descarga de RCSB (o pon `PDB_FILE` a un archivo subido). `IONIZE=True` protona a pH ~7 (`SITE=(IONIZE)`:
Arg/Lys +, Asp/Glu −, SO4/PO4 ionizados); por defecto los grupos quedan neutros, como en los benchmarks.
`EPS=78.4` activa agua implícita (COSMO); `None` = vacío. Tiempos orientativos en una A100: crambina (642 átomos)
~0.1 s por paso de dinámica (≈1 min por cada 50 fs), 1AKE (6689 átomos) ~0.9 s por paso.

In [ ]:
PDB_ID = '1CRN'          # código PDB a descargar
PDB_FILE = None          # o la ruta de un PDB ya subido (tiene prioridad sobre PDB_ID)
IONIZE = False           # True: SITE=(IONIZE), pH ~7
EPS = None               # 78.4 = agua implícita (COSMO); None = vacío
TEMPERATURE = 300.0      # K
OPT_CYCLES = 100         # ciclos de optimización
INTERVAL_FS = 0.5        # femtosegundos entre filas de la tabla DRC
NVT_POINTS = 200         # filas de NVT (200 x 0.5 fs = 100 fs)
NVE_POINTS = 200         # filas de NVE
BUSSI_FS = 100.0         # BUSSI=: constante de tiempo del termostato (fs)
SEED = 1                 # semilla de las velocidades y del termostato
USE_GPU = True           # False: misma corrida en CPU (MOPAC_NOGPU=1), para comparar

import sys
sys.path.insert(0, str(SRC / 'scripts'))
import importlib, mozyme_md_workflow as md
importlib.reload(md)
WORK = CONTENT / 'mozyme_md' / (PDB_FILE and Path(PDB_FILE).stem or PDB_ID)
ENV = None if USE_GPU else {'MOPAC_NOGPU': '1'}
print('work dir:', WORK)

## 5. Preparar el PDB

Limpia el PDB (aguas, conformaciones alternativas, grupos parciales en posiciones especiales del cristal), añade
hidrógenos con MOPAC `ADD-H` y pasa la comprobación química de MOPAC (`INPUT CHEMISTRY CHECK`): si hay errores
(grupos incompletos, cargas imposibles) la celda se detiene y los muestra.

In [ ]:
import urllib.request
WORK.mkdir(parents=True, exist_ok=True)
if PDB_FILE:
    raw = Path(PDB_FILE)
else:
    raw = WORK / f'{PDB_ID}.pdb'
    urllib.request.urlretrieve(f'https://files.rcsb.org/download/{PDB_ID}.pdb', raw)
HYDRO = md.prepare_pdb(MOPAC, raw, WORK / 'prepare', ionize=IONIZE)
for line in (WORK / 'prepare' / f'{raw.stem}_addh.out').read_text(errors='ignore').splitlines():
    if 'INPUT CHEMISTRY CHECK' in line or line.strip().startswith(('ERROR ', 'WARNING ', 'Errors:', 'No problems')):
        print(line.rstrip())

## 6. Optimización de geometría

In [ ]:
OPT_PDB, opt_run = md.optimize(MOPAC, HYDRO, WORK / 'opt', cycles=OPT_CYCLES, eps=EPS, env_extra=ENV)
print('optimized geometry:', OPT_PDB, ' heat', opt_run.heat)

## 7. Equilibrado NVT (termostato de Bussi)

Arranca de la geometría optimizada con velocidades de Maxwell-Boltzmann a `TEMPERATURE`. Gráficas: temperatura
instantánea, energías potencial/cinética/total y el error de integración.

In [ ]:
import matplotlib.pyplot as plt

def plot_dynamics(dyn, natoms, title):
    rows = dyn.rows
    t = [r[0] for r in rows]
    temps = md.temperatures(rows, natoms)
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    ax[0].plot(t, temps); ax[0].axhline(TEMPERATURE, ls='--', c='gray'); ax[0].set_xlabel('fs'); ax[0].set_ylabel('T (K)')
    ax[1].plot(t, [r[1] - rows[0][1] for r in rows], label='potencial')
    ax[1].plot(t, [r[2] for r in rows], label='cinética')
    ax[1].plot(t, [r[3] - rows[0][3] for r in rows], label='total')
    ax[1].set_xlabel('fs'); ax[1].set_ylabel('kcal/mol (relativo)'); ax[1].legend()
    ax[2].plot(t, [r[4] for r in rows]); ax[2].set_xlabel('fs'); ax[2].set_ylabel('ERROR (kcal/mol)')
    fig.suptitle(title); plt.tight_layout(); plt.show()
    tail = temps[len(temps) // 2:] or temps
    print(f'{title}: {len(rows)} puntos, {rows[-1][0]:.1f} fs en {dyn.run.wall:.1f} s; '
          f'T media (segunda mitad) {sum(tail) / len(tail):.1f} K; '
          f'max |ERROR| {max((abs(r[4]) for r in rows[3:]), default=0.0):.3f} kcal/mol; '
          f'SCF residente en GPU en todos los pasos: {dyn.run.gpu_ok}')

NATOMS = sum(1 for l in OPT_PDB.read_text().splitlines() if l.startswith(('ATOM', 'HETATM')))
nvt = md.dynamics(MOPAC, OPT_PDB, WORK / 'nvt', 'NVT', TEMPERATURE, NVT_POINTS, interval_fs=INTERVAL_FS,
                  tau_fs=BUSSI_FS, eps=EPS, seed=SEED, env_extra=ENV)
plot_dynamics(nvt, NATOMS, f'NVT {TEMPERATURE:g} K')

## 8. Producción NVE

Parte de la última configuración del NVT con velocidades nuevas de Maxwell-Boltzmann a la misma temperatura
(MOPAC no lee velocidades junto a una geometría PDB). En NVE la energía total debe mantenerse constante.

In [ ]:
START = md.last_frame_pdb(nvt.xyz, OPT_PDB, WORK / 'nvt' / 'nvt_last.pdb') if nvt.xyz else OPT_PDB
nve = md.dynamics(MOPAC, START, WORK / 'nve', 'NVE', TEMPERATURE, NVE_POINTS, interval_fs=INTERVAL_FS,
                  eps=EPS, seed=SEED + 1, env_extra=ENV)
plot_dynamics(nve, NATOMS, f'NVE (inicio a {TEMPERATURE:g} K)')

## 9. Agua implícita (COSMO) en la GPU: ¿corre residente?

Punto simple de la proteína preparada con `EPS=78.4` en CPU y en GPU: compara el calor de formación y muestra si el
SCF de la GPU fue residente (`success`) o volvió a la CPU (`fallback_cpu`).

In [ ]:
for label, env in (('CPU', {'MOPAC_NOGPU': '1'}), ('GPU', None)):
    r = md.run_mopac(MOPAC, WORK / 'cosmo' / label, 'cosmo', f'{md.BASE_KEYS} EPS=78.4 GEO_DAT="{OPT_PDB.name}" 1SCF',
                     'COSMO single point', [OPT_PDB], env)
    print(f'{label}: heat {r.heat}  wall {r.wall:.1f} s  statuses {sorted(set(r.statuses))}')

## 10. Visualizar la trayectoria y descargar

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py3Dmol'], check=False)
import py3Dmol
traj = nve.xyz or nvt.xyz
frames = md.xyz_frames(traj)
step = max(1, len(frames) // 50)
xyz_text = ''
for f in frames[::step]:
    xyz_text += f'{len(f)}\n\n' + ''.join(f'{e} {x:.4f} {y:.4f} {z:.4f}\n' for e, x, y, z in f)
view = py3Dmol.view(width=700, height=500)
view.addModelsAsFrames(xyz_text, 'xyz')
view.setStyle({'stick': {'radius': 0.12}})
view.animate({'loop': 'forward', 'interval': 80})
view.zoomTo()
view.show()

import shutil
archive = shutil.make_archive(str(CONTENT / f'mozyme_md_{WORK.name}'), 'zip', WORK)
print('archivo:', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print('descarga manual:', archive, exc)